In [15]:
!pip install papermill


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import os
import pandas as pd

In [17]:
# ==========================================
#  ZONE DE CONTRÔLE (À MODIFIER À LA MAIN)
# ==========================================

ma_region = 'Provence_Alpes_Cote_Azur'
region_code = 'PACA'
mes_departements = ['05','04','06','13','83','84']

In [18]:
import papermill as pm

def run_pipeline_for_region(nom_region, region_code, departements):
    print(f"Démarrage de la pipeline pour la région : {nom_region}")
    print(f"Départements à traiter : {departements}")
    
    # 1. Téléchargement pour chaque département
    for dep in departements:
        print(f"Téléchargement des données pour le département {dep}...")
        pm.execute_notebook(
            'DL_dep_DPE.ipynb',
            f'DL_dep_DPE_output_{dep}.ipynb', # Nom du notebook de sortie avec les logs
            parameters=dict(Dep=dep)
        )

    # ---  Affichage de la taille du dataset ---
        # On cherche le nom du fichier CSV sauvegardé par votre notebook
        nom_fichier = f'Dpe_dep_{dep}.csv'
        
        if os.path.exists(nom_fichier):
            # Astuce : on ne lit qu'une seule colonne (usecols=[0]) pour que ça soit instantané 
            # et ne pas saturer la mémoire de l'ordinateur
            taille_dataset = len(pd.read_csv(nom_fichier, usecols=[0]))
            print(f"   ---> ✅ Dataset {dep} prêt : {taille_dataset} logements récupérés.\n")
        else:
            print(f"   ---> ⚠️ Attention, le fichier {nom_fichier} n'a pas été trouvé après exécution.\n")
        
    # 2. Agrégation des données en une région
    print(f"Fusion des départements pour la région {nom_region}...")
    pm.execute_notebook(
        'get_data_region.ipynb',
        f'get_data_region_output_{nom_region}.ipynb',
        parameters=dict(region_code=region_code, departements=departements)
    )
    # ---  Affichage de la taille du dataset régional ---
    nom_fichier_region = f'dpe_{region_code}.csv'
    if os.path.exists(nom_fichier_region):
        taille_dataset_region = len(pd.read_csv(nom_fichier_region, usecols=[0]))
        print(f"   ---> 🌍 Dataset régional ({region_code}) prêt : {taille_dataset_region} logements au total réunis !\n")
    else:
        print(f"   ---> ⚠️ Attention, le fichier {nom_fichier_region} n'a pas été trouvé après la fusion.\n")

    # 3. Création du modèle électrique
    print(f"Création et entraînement du modèle électrique pour {nom_region}...")
    pm.execute_notebook(
        'creation_model_elec.ipynb',
        f'creation_model_elec_output_{nom_region}.ipynb',
        parameters=dict(nom_region=nom_region,region_code=region_code)
    )

     # 4. Création du modèle DPE
    print(f"Création et entraînement du modèle DPE pour {nom_region}...")
    pm.execute_notebook(
        'creation_model_dpe.ipynb',
        f'creation_model_dpe_output_{nom_region}.ipynb',
        parameters=dict(nom_region=nom_region,region_code=region_code)
    )
    
    print(f"✅ Pipeline terminée avec succès pour {nom_region} !")


# Lancement
run_pipeline_for_region(ma_region, region_code,mes_departements)

Démarrage de la pipeline pour la région : Provence_Alpes_Cote_Azur
Départements à traiter : ['05', '04', '06', '13', '83', '84']
Téléchargement des données pour le département 05...


Executing:   0%|          | 0/27 [00:00<?, ?cell/s]

Executing: 100%|██████████| 27/27 [07:58<00:00, 17.71s/cell]


   ---> ✅ Dataset 05 prêt : 36444 logements récupérés.

Téléchargement des données pour le département 04...


Executing: 100%|██████████| 27/27 [07:37<00:00, 16.94s/cell]


...

   ---> ✅ Dataset 06 prêt : 275897 logements récupérés.

Téléchargement des données pour le département 13...


Executing: 100%|██████████| 27/27 [1:05:16<00:00, 145.07s/cell]  


   ---> ✅ Dataset 13 prêt : 310783 logements récupérés.

Téléchargement des données pour le département 83...


Executing:  37%|███▋      | 10/27 [00:14<00:24,  1.47s/cell]


PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [9]":
---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
~\AppData\Local\Temp\ipykernel_15872\3798078458.py in ?()
      1 # 2. Lancement de la fonction d'analyse
----> 2 df_profil = profile_dataframe(dpe_dep)
      3 
      4 # 3. Affichage du résultat
      5 print("="*60)

~\AppData\Local\Temp\ipykernel_15872\4047656487.py in ?(df)
     50             return 'Texte (ID/Nom)'
     51 
     52         return str(dtype) # Type par défaut
     53 
---> 54     profile['Classe Inférencée'] = df.apply(determine_data_class)
     55 
     56     return profile.sort_values(by='Nombre de Catégories Uniques', ascending=False)

c:\Users\lilym\Documents\Capstone\Projet-DPE\Analyse\.venv\Lib\site-packages\pandas\core\frame.py in ?(self, key, value)
   4308             s

In [ ]:
import os # <-- Indispensable pour supprimer des fichiers

def delete_temp_files(nom_region, region_code, departements):
    
    #  ETAPE DE NETTOYAGE A LA TOUTE FIN :
    print("Nettoyage des fichiers temporaires...")
    
    # On supprime les fichiers des départements
    for dep in departements :
        if os.path.exists( f'DL_dep_DPE_output_{dep}.ipynb'):
            os.remove( f'DL_dep_DPE_output_{dep}.ipynb')

    # On supprime le fichier de la région
    if os.path.exists(f'get_data_region_output_{nom_region}.ipynb'):
        os.remove(f'get_data_region_output_{nom_region}.ipynb')
        
    # On supprime le fichier du modèle
    if os.path.exists(f'creation_model_elec_output_{nom_region}.ipynb'):
        os.remove(f'creation_model_elec_output_{nom_region}.ipynb')

    if os.path.exists(f'creation_model_dpe_output_{nom_region}.ipynb'):
        os.remove(f'creation_model_dpe_output_{nom_region}.ipynb')

        
    print("Pipeline terminée et dossier tout propre !")

In [ ]:
delete_temp_files(ma_region, region_code, mes_departements)

In [ ]:
def delete_dpe_files(nom_region, region_code, departements):
    
    #  ETAPE DE NETTOYAGE A LA TOUTE FIN :
    print("Nettoyage des fichiers temporaires...")
    
    # On supprime les fichiers des départements
    for dep in departements :
        if os.path.exists( f'Dpe_dep_{dep}.ipynb'):
            os.remove( f'Dpe_dep_{dep}.ipynb')

    # On supprime le fichier de la région
    if os.path.exists(f'dpe_{nom_region}.ipynb'):
        os.remove(f'dpe_{nom_region}.ipynb')
                
    print("Pipeline terminée et dossier tout propre !")